# Progressive Token Streaming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibm-granite-community/mellea-cookbook/blob/main/recipes/ProgressiveTokenStreaming/ProgressiveTokenStreaming.ipynb)

Large language models generate text one token at a time. By default many applications wait until the entire response is complete before displaying anything, which can mean several seconds of silence even for short answers. **Progressive token streaming** pipes each token to the user as it is produced, dramatically improving perceived responsiveness.

This recipe covers the full streaming stack:

1. **Mellea streaming primitives** — `ModelOption.STREAM` and `ModelOutputThunk.astream()` poll loop
2. **In-notebook real-time output** — progressive display without any server
3. **FastAPI SSE backend** — a production-grade `text/event-stream` endpoint backed by Mellea
4. **Interactive chat UI** — a ChatGPT-style widget rendered directly inside the notebook

**Prerequisites:** Ollama running locally with `granite4.1:3b` pulled (**Option A**), **or** IBM watsonx.ai credentials (**Option B**). Pick one backend below and skip the other.

## Step 1. Install dependencies

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install \
    "git+https://github.com/ibm-granite-community/utils.git" \
    "mellea[litellm]" \
    fastapi \
    "uvicorn[standard]" \
    nest_asyncio \
    httpx
! echo "::endgroup::"

## Step 2. Backend setup

Run **exactly one** of the two options below.

### Option A — Local Ollama (no API key required)

Pull required granite model by running `ollama pull granite4.1:3b` and start ollama server by running `ollama serve` . Skip this cell if you prefer watsonx.ai.

In [ ]:
import os
from mellea import start_session, MelleaSession
from mellea.stdlib.context import ChatContext

model_id = "granite4.1:3b"

m: MelleaSession = start_session(model_id=model_id, ctx=ChatContext())
print("Session ready (Ollama):", m)

### Option B — IBM watsonx.ai

See [Getting Started with IBM watsonx](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_WatsonX.ipynb) for credential setup. You need `WATSONX_URL`, `WATSONX_APIKEY`, and `WATSONX_PROJECT_ID`. Skip this cell if you are using Ollama.

In [ ]:
from ibm_granite_community.notebook_utils import get_env_var

# Load credentials into environment variables expected by LiteLLM
get_env_var("WATSONX_APIKEY")
get_env_var("WATSONX_PROJECT_ID")
get_env_var("WATSONX_URL")

In [ ]:
from mellea import start_session, MelleaSession
from mellea.stdlib.context import ChatContext

m: MelleaSession = start_session(
    backend_name="litellm",
    model_id="watsonx/ibm/granite-4-h-small",
    ctx=ChatContext(),
)
print("Session ready (watsonx.ai):", m)

---

## Step 3. Core Mellea streaming — `ModelOption.STREAM` and `astream()`

### Why streaming matters: Time to First Token (TTFT)

The latency between submitting a prompt and receiving the *first token* of the response is called **Time to First Token (TTFT)**. For a typical 200-token reply at 50 tokens/second, TTFT without streaming means ~4 seconds of silence. With streaming enabled the user sees the first word in well under a second.

### How Mellea exposes streaming

Mellea's async entry point `ainstruct()` returns a `ModelOutputThunk` immediately — the backend generation runs concurrently. Two options work together to enable streaming:

- `model_options={ModelOption.STREAM: True}` — keeps the HTTP connection open for incremental token delivery.
- `strategy=None` — bypasses the default `RejectionSamplingStrategy`, which would otherwise await the *entire* response before returning the thunk. Without this, `is_computed()` is already `True` on return and there is nothing left for the poll loop to consume.

Tokens are consumed by calling `await thunk.astream()` in a loop; each call returns the delta (new text) received since the previous call, until `thunk.is_computed()` signals the stream is finished.

> **`astream()` is a coroutine, not an async iterator.** Do not use `async for` on it; call it repeatedly in a `while not thunk.is_computed()` loop.

In [ ]:
import asyncio
from mellea.backends import ModelOption


async def stream_to_stdout(session: MelleaSession, prompt: str) -> str:
    """Stream a Mellea instruction, printing tokens as they arrive.

    Returns the fully assembled response string.
    """
    # strategy=None bypasses the sampling/retry wrapper so ainstruct() returns
    # a live, uncomputed thunk immediately. Without it the default
    # RejectionSamplingStrategy would await the full response before returning,
    # leaving nothing left for the astream() loop to consume.
    # astream() is a coroutine (not an async iterator) — call it in a loop
    # until is_computed() signals the stream is finished.
    thunk = await session.ainstruct(
        prompt,
        model_options={ModelOption.STREAM: True},
        strategy=None,
    )
    chunks: list[str] = []
    while not thunk.is_computed():
        delta = await thunk.astream()
        if delta:
            print(delta, end="", flush=True)
            chunks.append(delta)
    print()  # newline after stream ends
    return "".join(chunks)


response = await stream_to_stdout(
    m,
    "Explain how streaming is better than non streaming response",
)

The tokens appear progressively above. The variable `response` now holds the complete assembled text:

In [ ]:
print("Full response collected:", len(response), "characters")

### Comparison: non-streaming vs streaming

The next two cells time the same prompt both ways so you can see the TTFT difference.

In [ ]:
import time

PROMPT = "List five benefits of using typed Python in data pipelines."

# --- Non-streaming: wait for the complete response ---
t0 = time.perf_counter()
result = m.instruct(PROMPT)
# Resolving the thunk triggers the actual HTTP call
full_text = str(result)
non_stream_elapsed = time.perf_counter() - t0

print(f"Non-streaming total wait: {non_stream_elapsed:.2f}s")

In [ ]:
# --- Streaming: measure time to first token ---


async def measure_ttft(session: MelleaSession, prompt: str) -> tuple[float, float]:
    """Return (time_to_first_token, total_elapsed) in seconds."""
    t0 = time.perf_counter()
    thunk = await session.ainstruct(
        prompt,
        model_options={ModelOption.STREAM: True},
        strategy=None,
    )
    ttft = None
    while not thunk.is_computed():
        delta = await thunk.astream()
        if ttft is None and delta:
            ttft = time.perf_counter() - t0
    total = time.perf_counter() - t0
    return ttft or 0.0, total


ttft, total = await measure_ttft(m, PROMPT)
print(f"Streaming — TTFT: {ttft:.2f}s | total: {total:.2f}s")
print(f"Non-streaming total: {non_stream_elapsed:.2f}s")
print(
    f"\nStreaming delivers the first token at {ttft:.2f}s, {non_stream_elapsed - ttft:.2f}s sooner — "
    "a meaningful UX improvement for longer responses."
)

---

## Step 4. FastAPI Server-Sent Events (SSE) backend

### Server-Sent Events primer

**Server-Sent Events (SSE)** is a lightweight HTTP/1.1 protocol for server-to-client push over a persistent connection. The server sets `Content-Type: text/event-stream` and writes newline-delimited frames:

```
data: Hello\n\n
data: world\n\n
data: [DONE]\n\n
```

Each `data:` line followed by **two newlines** is one event. The client's `EventSource` API (or a `fetch` + `ReadableStream`) receives events as they arrive. SSE is ideal for chat streaming because:

- It works over plain HTTP (no WebSocket upgrade)
- Automatic reconnect is built into the browser protocol
- It is a one-way channel — exactly what LLM streaming needs

### Architecture

```
Browser / Notebook widget
        │  POST /chat  { "message": "..." }
        ▼
  FastAPI endpoint
        │  StreamingResponse(content=token_generator())
        ▼
  Mellea thunk.astream()  →  LLM backend (Ollama / watsonx.ai)
```


In [ ]:
import asyncio
import json
import threading
from contextlib import asynccontextmanager

import time
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel as PydanticModel

nest_asyncio.apply()

import asyncio.tasks as _asyncio_tasks
import asyncio.events as _asyncio_events

# -----------------------------------------------------------------------
# Python 3.14 + nest_asyncio compatibility shim
# -----------------------------------------------------------------------
# nest_asyncio replaces asyncio.Task with _PyTask (pure-Python). _PyTask
# tracks the running task in asyncio.tasks._current_tasks, but the C
# builtin asyncio.current_task() reads C-level state that _PyTask never
# updates. sniffio and anyio both call the C builtin, so they see None
# and raise AsyncLibraryNotFoundError / NoEventLoopError inside every
# anyio-backed primitive (FastAPI StreamingResponse, httpcore, etc.).
#
# Replace the C-level current_task reference in asyncio, sniffio and
# anyio with a Python wrapper that reads _current_tasks directly.
# asyncio.timeout() also calls the C builtin directly, so it too needs
# to be patched.

def _py_current_task(loop=None):
    """Python 3.14-safe current_task() that works with nest_asyncio's _PyTask."""
    if loop is None:
        try:
            loop = _asyncio_events.get_running_loop()
        except RuntimeError:
            return None
    return _asyncio_tasks._current_tasks.get(loop)


import sniffio._impl as _sniffio_impl


def _patched_current_async_library():
    v = _sniffio_impl.thread_local.name
    if v is not None:
        return v
    v = _sniffio_impl.current_async_library_cvar.get()
    if v is not None:
        return v
    if "asyncio" in __import__("sys").modules:
        try:
            if _py_current_task() is not None:
                return "asyncio"
        except Exception:
            pass
    raise _sniffio_impl.AsyncLibraryNotFoundError(
        "unknown async library, or not in async context"
    )


import sniffio as _sniffio

_sniffio_impl.current_async_library = _patched_current_async_library
_sniffio.current_async_library = _patched_current_async_library

import anyio._backends._asyncio as _anyio_asyncio

_anyio_asyncio.current_task = _py_current_task
# Also patch asyncio.current_task itself: asyncio.timeout() calls
# asyncio.tasks.current_task() (the C builtin) directly, which returns
# None for nest_asyncio _PyTask objects on Python 3.14, causing
# RuntimeError: Timeout should be used inside a task.
asyncio.current_task = _py_current_task
_asyncio_tasks.current_task = _py_current_task
# -----------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Request / response schema
# ---------------------------------------------------------------------------


class ChatRequest(PydanticModel):
    message: str


# ---------------------------------------------------------------------------
# FastAPI application
# ---------------------------------------------------------------------------

app = FastAPI(title="Mellea SSE Chat")

# Allow the in-notebook widget (different origin) to call the API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["POST", "GET", "OPTIONS"],
    allow_headers=["*"],
)


_STREAM_DONE = object()  # sentinel for the queue


async def _mellea_stream_task(
    message: str, queue: asyncio.Queue
) -> None:
    """Run the Mellea astream() poll loop inside a real asyncio.Task.

    asyncio.timeout() (used internally by mellea's send_to_queue) requires
    the call-site to be running inside an asyncio.Task. When astream() is
    called directly from an async-generator driven by Starlette's
    StreamingResponse the task context may be None under nest_asyncio +
    Python 3.14, causing ``RuntimeError: Timeout should be used inside a
    task``. Wrapping the poll loop in a dedicated task (via
    asyncio.create_task) guarantees a valid task context at all times.
    """
    try:
        thunk = await m.ainstruct(
            message,
            model_options={ModelOption.STREAM: True},
            strategy=None,
        )
        while not thunk.is_computed():
            delta = await thunk.astream()
            if delta:
                await queue.put(delta)
        await queue.put(_STREAM_DONE)
    except Exception as exc:
        await queue.put(exc)


async def token_generator(message: str):
    """Async generator that streams Mellea tokens as SSE frames."""
    queue: asyncio.Queue = asyncio.Queue()
    # Run the Mellea streaming inside a proper asyncio.Task so that
    # asyncio.timeout() inside mellea's send_to_queue has a task context.
    # Use get_running_loop() — NOT get_event_loop() — so the task
    # is created on the uvicorn request loop, not the Jupyter loop.
    task = asyncio.get_running_loop().create_task(
        _mellea_stream_task(message, queue)
    )
    try:
        while True:
            item = await queue.get()
            if item is _STREAM_DONE:
                break
            if isinstance(item, Exception):
                raise item
            # SSE format: "data: <json-encoded-payload>\n\n"
            # json.dumps escapes \n, \r and other control characters so the
            # payload is safe to embed on a single line — a bare \n inside
            # item would otherwise split the SSE frame and corrupt the stream.
            yield f"data: {json.dumps(item)}\n\n"
    finally:
        task.cancel()
    # Signal end-of-stream to the client
    yield "data: [DONE]\n\n"


@app.post("/chat")
async def chat(request: ChatRequest):
    """SSE endpoint: streams tokens for the given chat message."""
    return StreamingResponse(
        token_generator(request.message),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",  # disable Nginx proxy buffering
        },
    )


@app.get("/health")
async def health():
    return {"status": "ok"}


# ---------------------------------------------------------------------------
# Launch server in a background thread
# ---------------------------------------------------------------------------

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8080

config = uvicorn.Config(
    app,
    host=SERVER_HOST,
    port=SERVER_PORT,
    log_level="warning",
)
server = uvicorn.Server(config)


def _run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())


thread = threading.Thread(target=_run_server, daemon=True)
thread.start()

# Give the server a moment to bind before the next cells run
time.sleep(1)
print(f"SSE server listening on http://{SERVER_HOST}:{SERVER_PORT}")

### Verify the server with the health endpoint

In [ ]:
import httpx

with httpx.Client() as client:
    resp = client.get(f"http://{SERVER_HOST}:{SERVER_PORT}/health")
    print("Health check:", resp.json())


### Test the SSE stream with a Python client

The `httpx` client below sends a chat message and reads back the SSE frames, printing each token as it arrives — mirroring what a browser `EventSource` would do.

In [ ]:
def sse_client(message: str) -> str:
    """Send a message to the SSE endpoint and stream the response tokens.

    Returns the assembled full response string.
    """
    url = f"http://{SERVER_HOST}:{SERVER_PORT}/chat"
    chunks: list[str] = []

    with httpx.Client(timeout=120) as client:
        with client.stream("POST", url, json={"message": message}) as response:
            response.raise_for_status()
            for line in response.iter_lines():
                if not line.startswith("data:"):
                    continue
                raw = line[len("data:"):]
                if raw.startswith(" "):
                    raw = raw[1:]
                if raw == "[DONE]":
                    break
                # Payload is JSON-encoded; decode to recover the original token
                # including any embedded newlines that would otherwise corrupt
                # the SSE frame on the server side.
                token = json.loads(raw)
                print(token, end="", flush=True)
                chunks.append(token)

    print()  # trailing newline
    return "".join(chunks)


full = sse_client("What is Server-Sent Events and when should I use it?")
print(f"\n[received {len(full)} characters total]")


---

## Step 5. Interactive chat UI

### How the browser consumes SSE

Modern browsers can read a streaming HTTP response without `EventSource` using the **Fetch API** and a `ReadableStreamDefaultReader` with `TextDecoderStream`:

```javascript
const response = await fetch('/chat', {
  method: 'POST',
  headers: { 'Content-Type': 'application/json' },
  body: JSON.stringify({ message }),
});
const reader = response.body
  .pipeThrough(new TextDecoderStream())
  .getReader();

while (true) {
  const { value, done } = await reader.read();
  if (done) break;
  // parse SSE lines from `value` and append tokens to the UI
}
```

The widget below uses exactly this pattern, rendered with `IPython.display.HTML` directly in the notebook cell output.

In [ ]:
from IPython.display import HTML, display

_CHAT_UI = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    font-size: 14px;
    background: #f0f2f5;
    display: flex;
    justify-content: center;
    padding: 12px;
  }}
  #chat-root {{
    width: 100%;
    max-width: 680px;
    display: flex;
    flex-direction: column;
    gap: 8px;
  }}
  #messages {{
    background: #fff;
    border: 1px solid #d1d5db;
    border-radius: 10px;
    padding: 12px;
    min-height: 240px;
    max-height: 420px;
    overflow-y: auto;
    display: flex;
    flex-direction: column;
    gap: 10px;
  }}
  .bubble {{
    padding: 8px 12px;
    border-radius: 14px;
    line-height: 1.5;
    max-width: 85%;
    white-space: pre-wrap;
    word-break: break-word;
  }}
  .user-bubble {{
    background: #2563eb;
    color: #fff;
    align-self: flex-end;
    border-bottom-right-radius: 4px;
  }}
  .assistant-bubble {{
    background: #f3f4f6;
    color: #111827;
    align-self: flex-start;
    border-bottom-left-radius: 4px;
  }}
  /* Typing cursor pulse */
  .cursor {{
    display: inline-block;
    width: 7px;
    height: 14px;
    background: #6b7280;
    margin-left: 2px;
    vertical-align: text-bottom;
    animation: blink 0.85s step-start infinite;
  }}
  @keyframes blink {{
    50% {{ opacity: 0; }}
  }}
  #input-row {{
    display: flex;
    gap: 8px;
  }}
  #user-input {{
    flex: 1;
    padding: 8px 12px;
    border: 1px solid #d1d5db;
    border-radius: 8px;
    font-size: 14px;
    resize: none;
    outline: none;
  }}
  #user-input:focus {{ border-color: #2563eb; }}
  #send-btn {{
    padding: 8px 18px;
    background: #2563eb;
    color: #fff;
    border: none;
    border-radius: 8px;
    cursor: pointer;
    font-size: 14px;
  }}
  #send-btn:disabled {{ background: #93c5fd; cursor: default; }}
  #status {{
    font-size: 12px;
    color: #6b7280;
    min-height: 16px;
  }}
</style>
</head>
<body>
<div id="chat-root">
  <div id="messages">
    <div class="bubble assistant-bubble">Hi! I am powered by Granite via Mellea. Ask me anything.</div>
  </div>
  <div id="input-row">
    <textarea id="user-input" rows="2" placeholder="Type a message… (Enter to send)"></textarea>
    <button id="send-btn">Send</button>
  </div>
  <div id="status"></div>
</div>

<script>
(function () {{
  const API = 'http://{SERVER_HOST}:{SERVER_PORT}/chat';
  const messagesEl = document.getElementById('messages');
  const inputEl    = document.getElementById('user-input');
  const sendBtn    = document.getElementById('send-btn');
  const statusEl   = document.getElementById('status');

  function scrollBottom() {{
    messagesEl.scrollTop = messagesEl.scrollHeight;
  }}

  function addBubble(role, text) {{
    const div = document.createElement('div');
    div.className = 'bubble ' + (role === 'user' ? 'user-bubble' : 'assistant-bubble');
    div.textContent = text;
    messagesEl.appendChild(div);
    scrollBottom();
    return div;
  }}

  async function sendMessage() {{
    const text = inputEl.value.trim();
    if (!text) return;

    inputEl.value = '';
    sendBtn.disabled = true;
    statusEl.textContent = 'Connecting…';

    addBubble('user', text);

    // Create an empty assistant bubble with a typing cursor
    const aDiv = document.createElement('div');
    aDiv.className = 'bubble assistant-bubble';
    const cursor = document.createElement('span');
    cursor.className = 'cursor';
    aDiv.appendChild(cursor);
    messagesEl.appendChild(aDiv);
    scrollBottom();

    let buffer = '';
    try {{
      const response = await fetch(API, {{
        method: 'POST',
        headers: {{ 'Content-Type': 'application/json' }},
        body: JSON.stringify({{ message: text }}),
      }});

      if (!response.ok) throw new Error('HTTP ' + response.status);
      statusEl.textContent = 'Streaming…';

      const reader = response.body
        .pipeThrough(new TextDecoderStream())
        .getReader();

      let leftovers = '';
      while (true) {{
        const {{ value, done }} = await reader.read();
        if (done) break;

        const raw = leftovers + value;
        const lines = raw.split('\\n');
        leftovers = lines.pop();  // last element may be incomplete

        for (const line of lines) {{
          if (!line.startsWith('data:')) continue;
          const raw5 = line.slice(5);
          const encoded = raw5.startsWith(" ") ? raw5.slice(1) : raw5;
          if (encoded === '[DONE]') break;
          // Payload is JSON-encoded; JSON.parse recovers the original token
          // text including any newlines that were escaped on the server side.
          buffer += JSON.parse(encoded);
          // Update bubble text, keeping cursor at the end
          aDiv.textContent = buffer;
          aDiv.appendChild(cursor);
          scrollBottom();
        }}
      }}

      statusEl.textContent = '';
    }} catch (err) {{
      statusEl.textContent = 'Error: ' + err.message;
      buffer = buffer || '(no response)';
    }}

    // Remove cursor, finalise bubble
    if (cursor.parentNode) cursor.parentNode.removeChild(cursor);
    aDiv.textContent = buffer;
    sendBtn.disabled = false;
    inputEl.focus();
  }}

  sendBtn.addEventListener('click', sendMessage);
  inputEl.addEventListener('keydown', function (e) {{
    if (e.key === 'Enter' && !e.shiftKey) {{
      e.preventDefault();
      sendMessage();
    }}
  }});
}})();
</script>
</body>
</html>
"""

display(HTML(_CHAT_UI))

Type a message in the text area above and press **Enter** or click **Send**. Tokens will appear in the assistant bubble one by one as the model generates them.

> **Note:** The server runs only while the notebook kernel is alive. Restarting the kernel will shut it down — simply re-run Step 4 to restart it.

---

## Summary and best practices

### Stream lifecycle

```
client request
    │
    ▼
FastAPI receives POST /chat
    │
    ▼
StreamingResponse wraps async generator
    │  (HTTP 200 sent immediately — headers only)
    ▼
ainstruct() + astream() poll loop → LLM backend
    │
    ├─ token received → "data: <token>\n\n" flushed to client
    ├─ token received → ...
    │
    └─ stream exhausted → "data: [DONE]\n\n" → generator returns
```

### Error handling

- **Network interruption:** wrap the `astream()` loop in `try/except` and yield an `event: error\ndata: ...\n\n` SSE frame so the client can surface it.
- **Model timeout:** set a `timeout` on the `httpx.AsyncClient` (or LiteLLM config) and handle `asyncio.TimeoutError`.
- **Client disconnects:** FastAPI detects a dropped connection and the generator is garbage-collected. No special handling is required on the server side.

### Cancellation

To let the user abort a generation mid-stream, the front-end can call `reader.cancel()` and the back-end generator will be closed on the next iteration when `StreamingResponse` detects the disconnected client.

### Tradeoffs

| | Blocking (non-streaming) | Streaming |
|---|---|---|
| Time to first token | Full generation time | < 1 s |
| Implementation complexity | Low | Moderate |
| Server memory | Low | Low (generator-based) |
| Client complexity | Trivial | Fetch + SSE parser |
| Ideal for | Short, latency-tolerant replies | Interactive chat, long answers |

### Next steps

- Explore [Instruct-Validate-Repair](../InstructValidateRepair/InstructValidateRepair.ipynb) to add structured output and automatic retries to your streaming pipeline.
- See [Extracting Structured Data](../StructuredDataExtraction/StructuredDataExtraction.ipynb) for typed extraction patterns using `@generative` stubs.